<a href="https://colab.research.google.com/github/isti2415/CSC551-Data-Mining/blob/main/Assignment%202/Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 2: Python / NumPy basics

1. **Part A (2D and 3D):** centring, projection onto a line, projection onto a plane (Lecture 2, slides 6–10).
2. **Part B (numeric data analysis):** mean vector, covariance matrix and correlation of the wheat seeds dataset
   (Lecture 2.2).

Notation from the lectures: $\mathbf{D}$ is the $n \times d$ data matrix, $\mathbf{1} \in \mathbb{R}^n$ is the
vector of ones, and $\hat{\boldsymbol{\mu}} = \frac{1}{n}\mathbf{D}^T\mathbf{1}$ is the mean vector.

In [ ]:
%pip install -q pandas numpy plotly nbformat

In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

---
# Part A: Dataset download and characteristics

## Step 1: Download a dataset

> Download any dataset of your choice.

We use the **Iris** dataset from the UCI repository, the same dataset used in the lecture slides, so our plots can be
compared directly with slides 6 and 10. It has 150 flowers, 4 numeric attributes (in cm) and 3 classes.

In [3]:
iris_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
iris_cols = ["sepal length", "sepal width", "petal length", "petal width", "class"]
iris = pd.read_csv(iris_url, header=None, names=iris_cols).dropna()
print(iris.shape)
print(iris["class"].value_counts())
iris.head()

(150, 5)
class
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
Name: count, dtype: int64


,sepal length,sepal width,petal length,petal width,class
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


## Step 2: Read with pandas and convert to a NumPy array

`D` holds the 4 numeric attributes. The string class names are turned into numbers 0, 1, 2 in `y`, and
`class_names[k]` gives the name of class `k`.

In [4]:
D = iris.iloc[:, :4].to_numpy(dtype=float)
class_names, y = np.unique(iris["class"].to_numpy(), return_inverse=True)
class_names = list(class_names)
attr = iris_cols[:4]

print("D shape:", D.shape)
print("Classes:", class_names)

D shape: (150, 4)
Classes: ['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']


## Step 3: Plotly for interactive plots

> You should use plotly library to create interactive plots so that we can rotate the plot.

Each class gets its own shape, decided only by its class label `k` (like slide 10: squares, circles, triangles).
Plotly 3D markers have no triangle, so the 3D plots use circle / square / diamond.

In [5]:
symbols_2d = ["square", "circle", "triangle-up"]
symbols_3d = ["square", "circle", "diamond"]
colors = ["#636EFA", "#EF553B", "#00CC96"]

## 2D

### Step 4: Plot the first two dimensions

> Plot the data points using the first two dimensions. Use different shapes to plot datapoints for your different
> classes (see slide 6 and slide 10).

In [6]:
X2 = D[:, :2]

def scatter_2d(fig, P, size, name_suffix="", outline=None, showlegend=True):
    for k, name in enumerate(class_names):
        pts = P[y == k] if len(P) == len(y) else P[[k]]
        fig.add_trace(go.Scatter(
            x=pts[:, 0], y=pts[:, 1], mode="markers", name=name + name_suffix,
            showlegend=showlegend,
            marker=dict(symbol=symbols_2d[k], size=size, color=colors[k],
                        line=dict(color=outline or "white", width=2 if outline else 0.5))))

fig = go.Figure()
scatter_2d(fig, X2, size=9)
fig.update_layout(title="Iris: " + attr[0] + " vs " + attr[1],
                  xaxis_title="X1: " + attr[0] + " (cm)", yaxis_title="X2: " + attr[1] + " (cm)",
                  width=850, height=550)
fig.show()

### Step 5: Class means on the same plot

> Calculate the mean data point for each class and show them with similar shape with the larger size over the above
> plot.

For class $k$ with data matrix $\mathbf{D}_k$ ($n_k$ rows): $\hat{\boldsymbol{\mu}}_k = \frac{1}{n_k}\mathbf{D}_k^T\mathbf{1}$.

In [7]:
def class_means(M):
    return np.array([M[y == k].T @ np.ones((y == k).sum()) / (y == k).sum()
                     for k in range(len(class_names))])

means2 = class_means(X2)
print(pd.DataFrame(means2, index=class_names, columns=attr[:2]))

scatter_2d(fig, means2, size=24, name_suffix=" mean", outline="black")
fig.update_layout(title="Iris with class means (large markers)")
fig.show()

                 sepal length  sepal width
Iris-setosa             5.006        3.418
Iris-versicolor         5.936        2.770
Iris-virginica          6.588        2.974


### Step 6: Centred data

> Calculate the centered version of the first two dimensions using vector formula. Plot them in a new plot with title:
> "Centered data"

$\mathbf{Z} = \mathbf{D} - \mathbf{1}\cdot\hat{\boldsymbol{\mu}}^T$ (Lecture 2, slide 7). The centred data has
mean $\mathbf{0}$, so the cloud is moved to the origin without changing its shape.

In [8]:
n = X2.shape[0]
ones = np.ones(n)
mu2 = X2.T @ ones / n
Z2 = X2 - np.outer(ones, mu2)

print("mean vector:", mu2)
print("mean of centred data:", np.round(Z2.T @ ones / n, 12))

fig = go.Figure()
scatter_2d(fig, Z2, size=9)
fig.update_layout(title="Centered data", xaxis_title="X1 (centred)", yaxis_title="X2 (centred)",
                  xaxis=dict(zeroline=True, zerolinecolor="black"),
                  yaxis=dict(zeroline=True, zerolinecolor="black", scaleanchor="x"),
                  width=850, height=600)
fig.show()

mean vector: [5.84333333 3.054     ]
mean of centred data: [ 0. -0.]


### Step 7: The line $\ell$

> Now, plot a line ($\ell$) in this plot with line equation. The line is $\ell = \operatorname{span}\{(-1.75,\ 1.75)^T\}$.
> Therefore, the equation of the line is $x_1 = -x_2$ (see slide 10).

$\ell$ is every multiple $t\,\mathbf{a}$ of $\mathbf{a} = (-1.75, 1.75)^T$. It goes through the origin, which is
where the centred data now sits (as on slide 10), so we draw it on the centred plot.

In [9]:
a = np.array([-1.75, 1.75])
t = np.linspace(-1.3, 1.3, 2)
line = np.outer(t, a)

fig.add_trace(go.Scatter(x=line[:, 0], y=line[:, 1], mode="lines", name="line ℓ: x1 = −x2",
                         line=dict(color="black", width=2)))
fig.add_annotation(x=line[0, 0], y=line[0, 1], text="ℓ: x<sub>1</sub> = −x<sub>2</sub>",
                   showarrow=False, xanchor="left", yanchor="bottom", font=dict(size=16))
fig.update_layout(title="Centered data with the line ℓ = span{(−1.75, 1.75)}")
fig.show()

### Step 8: Project every point onto $\ell$

> Now calculate the projection of each data point on the line $\ell$. And plot the projected point on the line using
> the same shape but smaller size. So all smaller shapes would be on the line (see slide 10).

Orthogonal projection of $\mathbf{z}$ onto $\mathbf{a}$ (Lecture 2, slide 9): $\mathbf{p} = \left(\frac{\mathbf{a}^T\mathbf{z}}{\mathbf{a}^T\mathbf{a}}\right)\mathbf{a}$.

For all points at once: $\mathbf{P} = \frac{1}{\mathbf{a}^T\mathbf{a}}\,(\mathbf{Z}\mathbf{a})\,\mathbf{a}^T$.
The dotted lines join every point to its projection, as on slide 10. They are perpendicular to $\ell$.

In [10]:
P2 = np.outer(Z2 @ a, a) / (a @ a)

# checks: every projection lies on x1 = -x2, and the residual z - p is orthogonal to a
print("max |p1 + p2|:", np.abs(P2[:, 0] + P2[:, 1]).max())
print("max |(z - p)·a|:", np.abs((Z2 - P2) @ a).max())

# dotted lines from each point to its projection (None breaks the line between segments)
seg = np.full((3 * n, 2), np.nan)
seg[0::3], seg[1::3] = Z2, P2
fig.add_trace(go.Scatter(x=seg[:, 0], y=seg[:, 1], mode="lines", name="projection",
                         line=dict(color="gray", width=1, dash="dot"), hoverinfo="skip"))
scatter_2d(fig, P2, size=6, name_suffix=" projected")
fig.update_layout(title="Centered data projected onto ℓ: x1 = −x2")
fig.show()

max |p1 + p2|: 0.0
max |(z - p)·a|: 6.661338147750939e-16


## 3D

### Steps 1–3: First three dimensions, one shape per class, with class means

In [11]:
X3 = D[:, :3]
means3 = class_means(X3)
print(pd.DataFrame(means3, index=class_names, columns=attr[:3]))

def scatter_3d(fig, P, size, name_suffix="", outline=None):
    for k, name in enumerate(class_names):
        pts = P[y == k] if len(P) == len(y) else P[[k]]
        fig.add_trace(go.Scatter3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode="markers", name=name + name_suffix,
            marker=dict(symbol=symbols_3d[k], size=size, color=colors[k],
                        line=dict(color=outline or colors[k], width=3 if outline else 0))))

fig = go.Figure()
scatter_3d(fig, X3, size=4)
scatter_3d(fig, means3, size=12, name_suffix=" mean", outline="black")
fig.update_layout(title="Iris: first three dimensions with class means (large markers)",
                  scene=dict(xaxis_title="X1: " + attr[0], yaxis_title="X2: " + attr[1],
                             zaxis_title="X3: " + attr[2]),
                  width=900, height=700)
fig.show()

                 sepal length  sepal width  petal length
Iris-setosa             5.006        3.418         1.464
Iris-versicolor         5.936        2.770         4.260
Iris-virginica          6.588        2.974         5.552


### Step 4: Centred data in 3D

> Calculate the centered version of the first three dimensions. Plot centered data points in 3D in a new plot.

In [12]:
mu3 = X3.T @ ones / n
Z3 = X3 - np.outer(ones, mu3)
print("mean vector:", mu3)

def centred_3d_figure():
    fig = go.Figure()
    scatter_3d(fig, Z3, size=5)
    fig.update_layout(title="Centered data (3D)",
                      scene=dict(xaxis_title="X1", yaxis_title="X2", zaxis_title="X3",
                                 aspectmode="data"),
                      width=900, height=750)
    return fig

centred_3d_figure().show()

mean vector: [5.84333333 3.054      3.75866667]


### Step 5: The plane spanned by $\mathbf{u} = (1, -2, 1)^T$ and $\mathbf{v} = (2, 1, 0)^T$

> Now, plot the plane (yellow plane) spanned by the two vectors on the plot done in step 4.

Every point of the plane is $s\,\mathbf{u} + t\,\mathbf{v}$. $\mathbf{u}^T\mathbf{v} = 2 - 2 + 0 = 0$, so the two
vectors are orthogonal. The plane's normal is $\mathbf{u} \times \mathbf{v} = (-1, 2, 5)^T$, so its equation is
$-x_1 + 2x_2 + 5x_3 = 0$.

In [13]:
u = np.array([1.0, -2.0, 1.0])
v = np.array([2.0, 1.0, 0.0])
A = np.column_stack([u, v])                   # 3 x 2 basis of the plane
print("u·v =", u @ v, "  normal u×v =", np.cross(u, v))

s, t = np.meshgrid(np.linspace(-1.5, 1.5, 2), np.linspace(-1.5, 1.5, 2))
plane = s[..., None] * u + t[..., None] * v   # grid of points s*u + t*v

def add_plane(fig):
    fig.add_trace(go.Surface(x=plane[..., 0], y=plane[..., 1], z=plane[..., 2],
                             colorscale=[[0, "yellow"], [1, "yellow"]], opacity=0.5,
                             showscale=False, name="plane span{u, v}", showlegend=True,
                             hoverinfo="skip"))

fig = centred_3d_figure()
add_plane(fig)
fig.update_layout(title="Centered data with the plane span{(1, −2, 1), (2, 1, 0)}: −x1 + 2x2 + 5x3 = 0")
fig.show()

u·v = 0.0   normal u×v = [-1.  2.  5.]


### Step 6: Project every point onto the plane

> Now calculate the projection of each data point on the plane. And plot the projected point on the plane using the
> same shape but smaller size in the plot done in step 4.

Projecting onto the column space of $\mathbf{A} = [\mathbf{u}\ \mathbf{v}]$ uses the projection matrix
$\mathbf{P}_A = \mathbf{A}(\mathbf{A}^T\mathbf{A})^{-1}\mathbf{A}^T$. Each centred point $\mathbf{z}_i$ becomes
$\mathbf{P}_A\mathbf{z}_i$, i.e. for all points $\mathbf{P} = \mathbf{Z}\,\mathbf{P}_A^T$. Since $\mathbf{u} \perp \mathbf{v}$
this is the same as adding the two line projections
$\frac{\mathbf{u}^T\mathbf{z}}{\mathbf{u}^T\mathbf{u}}\mathbf{u} + \frac{\mathbf{v}^T\mathbf{z}}{\mathbf{v}^T\mathbf{v}}\mathbf{v}$.

In [14]:
P_A = A @ np.linalg.inv(A.T @ A) @ A.T
P3 = Z3 @ P_A.T

normal = np.cross(u, v)
print("projection matrix:\n", np.round(P_A, 4))
print("max |normal·p| (0 means on the plane):", np.abs(P3 @ normal).max())
print("max |(z - p)·u|, |(z - p)·v|:", np.abs((Z3 - P3) @ u).max(), np.abs((Z3 - P3) @ v).max())
print("same as sum of line projections:",
      np.allclose(P3, np.outer(Z3 @ u, u) / (u @ u) + np.outer(Z3 @ v, v) / (v @ v)))

fig = centred_3d_figure()
add_plane(fig)
seg3 = np.full((3 * n, 3), np.nan)
seg3[0::3], seg3[1::3] = Z3, P3
fig.add_trace(go.Scatter3d(x=seg3[:, 0], y=seg3[:, 1], z=seg3[:, 2], mode="lines", name="projection",
                           line=dict(color="gray", width=1, dash="dot"), hoverinfo="skip"))
for k, name in enumerate(class_names):
    pts = P3[y == k]
    fig.add_trace(go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode="markers",
                               name=name + " projected",
                               marker=dict(symbol=symbols_3d[k], size=3, color=colors[k])))
fig.update_layout(title="Centered data projected onto the yellow plane")
fig.show()

projection matrix:
 [[ 0.9667  0.0667  0.1667]
 [ 0.0667  0.8667 -0.3333]
 [ 0.1667 -0.3333  0.1667]]
max |normal·p| (0 means on the plane): 6.661338147750939e-16
max |(z - p)·u|, |(z - p)·v|: 6.661338147750939e-16 5.551115123125783e-16
same as sum of line projections: True


---
# Part B: Numeric data analysis

## Steps 1–3: Wheat seeds dataset

> Download wheat seeds dataset. Read the dataset using panda data frame. Convert to numpy array. Discard the last class
> label column.

210 wheat kernels, 7 measured attributes and a class label (Kama, Rosa, Canadian). Some rows of the file are separated
by more than one tab, so we split on any whitespace.

In [15]:
seeds_url = "http://archive.ics.uci.edu/ml/machine-learning-databases/00236/seeds_dataset.txt"
seeds_cols = ["area", "perimeter", "compactness", "kernel length", "kernel width",
              "asymmetry coefficient", "groove length", "class"]
seeds = pd.read_csv(seeds_url, sep=r"\s+", header=None, names=seeds_cols)
print(seeds.shape)
seeds.head()

(210, 8)


,area,perimeter,compactness,kernel length,kernel width,asymmetry coefficient,groove length,class
0,15.26,14.84,0.8710,5.763,3.312,2.221,5.220,1
1,14.88,14.57,0.8811,5.554,3.333,1.018,4.956,1
2,14.29,14.09,0.9050,5.291,3.337,2.699,4.825,1
3,13.84,13.94,0.8955,5.324,3.379,2.259,4.805,1
4,16.14,14.99,0.9034,5.658,3.562,1.355,5.175,1


In [16]:
S = seeds.to_numpy()[:, :-1].astype(float)   # drop the class label column
seed_attr = seeds_cols[:-1]
n_s, d_s = S.shape
print("data matrix:", S.shape)

data matrix: (210, 7)


## Question 1: Multivariate mean vector

$\hat{\boldsymbol{\mu}} = \frac{1}{n}\sum_{i=1}^n \mathbf{x}_i = \frac{1}{n}\mathbf{D}^T\mathbf{1}$

In [17]:
ones_s = np.ones(n_s)
mu_s = S.T @ ones_s / n_s

print(pd.DataFrame({"mean (vector formula)": mu_s, "np.mean check": S.mean(axis=0)}, index=seed_attr))

                       mean (vector formula)  np.mean check
area                               14.847524      14.847524
perimeter                          14.559286      14.559286
compactness                         0.870999       0.870999
kernel length                       5.628533       5.628533
kernel width                        3.258605       3.258605
asymmetry coefficient               3.700201       3.700201
groove length                       5.408071       5.408071


## Question 2: Sample covariance matrix as inner products of centred columns

Centred data matrix $\overline{\mathbf{D}} = \mathbf{D} - \mathbf{1}\cdot\hat{\boldsymbol{\mu}}^T$ with columns
$\overline{X}_1, \dots, \overline{X}_d$. Eq. (2.38):

$$\widehat{\boldsymbol{\Sigma}} = \frac{1}{n}\overline{\mathbf{D}}^T\overline{\mathbf{D}} =
\frac{1}{n}\begin{pmatrix} \overline{X}_1^T\overline{X}_1 & \cdots & \overline{X}_1^T\overline{X}_d \\
\vdots & \ddots & \vdots \\ \overline{X}_d^T\overline{X}_1 & \cdots & \overline{X}_d^T\overline{X}_d \end{pmatrix}$$

Entry $(j, k)$ is the inner product of centred columns $j$ and $k$, divided by $n$. $\overline{\mathbf{D}}^T\overline{\mathbf{D}}$
computes all of these inner products in one matrix product. `np.cov(..., bias=True)` (which also divides by $n$) is
only a check.

In [18]:
D_bar = S - np.outer(ones_s, mu_s)
Sigma = D_bar.T @ D_bar / n_s

pd.set_option("display.precision", 4)
print("Sample covariance matrix:")
print(pd.DataFrame(Sigma, index=seed_attr, columns=seed_attr))
print("\nsymmetric:", np.allclose(Sigma, Sigma.T))
print("matches np.cov(bias=True):", np.allclose(Sigma, np.cov(S, rowvar=False, bias=True)))
print("one entry by hand, X̄1·X̄2 / n =", D_bar[:, 0] @ D_bar[:, 1] / n_s, " Sigma[0,1] =", Sigma[0, 1])
print("total variance tr(Sigma) =", np.trace(Sigma))

Sample covariance matrix:
                         area  perimeter  compactness  kernel length  \
area                   8.4260     3.7605       0.0416         1.2189   
perimeter              3.7605     1.6974       0.0163         0.5600   
compactness            0.0416     0.0163       0.0006         0.0038   
kernel length          1.2189     0.5600       0.0038         0.1954   
kernel width           1.0618     0.4638       0.0068         0.1433   
asymmetry coefficient -0.9996    -0.4247      -0.0117        -0.1137   
groove length          1.2293     0.5690       0.0026         0.2022   

                       kernel width  asymmetry coefficient  groove length  
area                         1.0618                -0.9996         1.2293  
perimeter                    0.4638                -0.4247         0.5690  
compactness                  0.0068                -0.0117         0.0026  
kernel length                0.1433                -0.1137         0.2022  
kernel width     

## Question 3: Correlation of attributes 1 and 2 as the cosine of an angle

Lecture 2.2, slide 19: the correlation is the cosine of the angle between the centred attribute vectors.

$$\hat{\rho}_{12} = \frac{\overline{X}_1^T\overline{X}_2}{\|\overline{X}_1\|\,\|\overline{X}_2\|} =
\left(\frac{\overline{X}_1}{\|\overline{X}_1\|}\right)^T\left(\frac{\overline{X}_2}{\|\overline{X}_2\|}\right) = \cos\theta$$

In [19]:
X1_bar, X2_bar = D_bar[:, 0], D_bar[:, 1]
cos_theta = (X1_bar @ X2_bar) / (np.sqrt(X1_bar @ X1_bar) * np.sqrt(X2_bar @ X2_bar))

print("correlation (cos theta) =", cos_theta)
print("angle theta =", np.degrees(np.arccos(cos_theta)), "degrees")
print("np.corrcoef check =", np.corrcoef(S[:, 0], S[:, 1])[0, 1])

correlation (cos theta) = 0.9943409310122261
angle theta = 6.098389313799515 degrees
np.corrcoef check = 0.9943409310122265


In [20]:
fig = go.Figure(go.Scatter(x=S[:, 0], y=S[:, 1], mode="markers",
                           marker=dict(size=7, color="#636EFA", line=dict(color="white", width=0.5)),
                           name="kernel"))
fig.update_layout(title=f"Wheat seeds: {seed_attr[0]} vs {seed_attr[1]}  (ρ = cos θ = {cos_theta:.4f})",
                  xaxis_title="X1: area", yaxis_title="X2: perimeter",
                  width=800, height=550)
fig.show()

The correlation is about 0.994 and the angle between the two centred vectors is only about 6°. That means a very strong
positive linear relation, which the scatter plot shows: the points lie almost on a straight line. This makes sense,
since bigger kernels have both a larger area and a larger perimeter.